In [98]:
import pandas as pd
import numpy as np

In [ ]:
TRAIN_DATA_PATH = "train.csv"
TEST_DATA_PATH = "test.csv"
first_hidden_layer_neurons = 1000
second_hidden_layer_neurons = 500 
k = 5
no_of_classes = 10
alpha  = 0.01
learning_rate = 0.001
epochs = 10
model_file_name = "final_model_weights.npz"

In [100]:
data = pd.read_csv(TRAIN_DATA_PATH)
data = np.array(data)

def spiltData(data :np.array):
    return data[0:41000] , data[41000:]


In [101]:
rand = np.random.default_rng(seed=42)
w1 = rand.random((first_hidden_layer_neurons , 784))*0.01
w2 = rand.random((second_hidden_layer_neurons , first_hidden_layer_neurons))*0.01
w3 = rand.random(( no_of_classes , second_hidden_layer_neurons)) * 0.01
b1 = np.ones(first_hidden_layer_neurons)
b2 = np.ones(second_hidden_layer_neurons)
b3 = np.ones(no_of_classes)

In [102]:
def leaky_relu(z: np.array):
    return np.where(z > 0 , z , alpha * z)

def softmax(z:np.array):
    z = z - np.max(z) 
    exp = np.exp(z)
    result = exp/exp.sum()
    return result.reshape(-1,1)

def oneHotEncoding(label ):
    y = np.zeros(no_of_classes)
    y[label] = 1
    return y

def act_tanh(z : np.array):
    return np.tanh(z)

def grad_tanh(z : np.array):
    return 1 - (np.tanh(z) ** 2)

def grad_relu(z:np.array):
    return np.where(z > 0 , 1 , alpha)

def save_model(filepath = "fnn_12_neuron_weights.npz"):
    np.savez(filepath, w1=w1, b1=b1, w2=w2, b2=b2, w3=w3, b3=b3)
    print(f"Model saved to {filepath}")

def load_model(filepath="fnn_12_neuron_weights.npz"):
    global w1, w2, b1, b2 , w3 , b3
    data = np.load(filepath)
    w1 = data['w1']
    b1 = data['b1']
    w2 = data['w2']
    b2 = data['b2']
    w3 = data['w3']
    b3 = data['b3']
    print(f"Model loaded from {filepath}")
    
def entropy_loss(y:np.array , y_hat):
   return -np.sum(y * np.log(y_hat + 1e-9))

In [103]:
def forwardPropgagation(x : np.array ):
    global w1, w2, b1, b2 , w3 , b3
    x = x.reshape(-1,1) 
    z1 = w1 @ x + b1.reshape(-1,1)
    a1 = leaky_relu(z1) 
    z2 = w2 @ a1 + b2.reshape(-1, 1)
    a2 = act_tanh(z2)
    z3 = w3 @ a2 + b3.reshape(-1,1) 
    y_hat = softmax(z3)
    return y_hat , a1, a2 , z1, z2 
    

def backwordPropagation(y_hat ,label , a1:np.array , a2 : np.array , z1 , z2 ,x  ):
    global w1, w2, b1, b2 , w3 , b3
    y_actual = oneHotEncoding(label=label)
    y_actual= y_actual.reshape(-1,1)
    dlz3 =  y_hat - y_actual
    dlb3 = dlz3.flatten()
    dlw3 = dlz3 @ a2.T
    dla2 = w3.T @ dlz3
    dlz2 = dla2 * grad_tanh(z2)
    dlw2 = dlz2 @ a1.T 
    dlb2 = dlz2.flatten()
    dla1 = w2.T @ dlz2
    dlz1 = dla1 * grad_relu(z1)
    dlw1 = dlz1 @ x.reshape(1, -1)   
    dlb1 = dlz1.flatten()

    w1 = w1 - (learning_rate * dlw1)
    b1 = b1 - (learning_rate * dlb1)
    w2 = w2 - (learning_rate * dlw2)
    b2 = b2 - (learning_rate * dlb2)
    w3 = w3 - (learning_rate * dlw3)
    b3 = b3 - (learning_rate * dlb3)
    loss = entropy_loss(y_actual , y_hat=y_hat)
    return loss

In [ ]:
def k_fold(X, y):
    """
    Splits data into k folds for cross-validation
    Returns list of (train_indices, val_indices) tuples
    """
    indices = np.random.permutation(len(X))
    folds = np.array_split(indices, k)
    
    fold_splits = []
    for i in range(k):
        val_indices = folds[i]
        train_indices = np.concatenate([folds[j] for j in range(k) if j != i])
        fold_splits.append((train_indices, val_indices))
    
    return fold_splits




def evaluate(X: np.array, y: np.array):
    """Evaluates accuracy and loss on given dataset"""
    correct = 0
    total_loss = 0
    
    for i in range(len(X)):
        sample = X[i, 1:] / 255.0 
        label = int(X[i, 0])
        
        y_hat, _, _, _, _  = forwardPropgagation(sample)
        y_actual = oneHotEncoding(label)
        total_loss += entropy_loss(y_actual, y_hat)
        
        if np.argmax(y_hat) == label:
            correct += 1
    
    accuracy = correct / len(X)
    avg_loss = total_loss / len(X)
    return accuracy, avg_loss


def train_network():
    global w1, w2, b1, b2, w3, b3
    
    X = pd.read_csv(TRAIN_DATA_PATH)
    X = np.array(X)
    
    train_data, val_data = spiltData(X)
    
    fold_splits = k_fold(train_data, train_data[:, 0])
    
    fold_accuracies = []
    fold_losses = []
    best_accuracy = 0
    best_weights = None
    best_fold = 0
    
    # K-Fold Cross-Validation
    for fold_num, (train_idx, val_idx) in enumerate(fold_splits):
        print(f"\n{'='*50}")
        print(f"Fold {fold_num + 1}/{k}")
        print(f"{'='*50}")
        
        # Reset weights for each fold
        rand = np.random.default_rng(seed=42)
        w1 = rand.random((first_hidden_layer_neurons , 784)) * 0.01
        w2 = rand.random((second_hidden_layer_neurons , first_hidden_layer_neurons)) * 0.01
        w3 = rand.random((no_of_classes , second_hidden_layer_neurons)) * 0.01
        b1 = np.ones(first_hidden_layer_neurons)
        b2 = np.ones(second_hidden_layer_neurons)
        b3 = np.ones(no_of_classes)
        
        fold_train_data = train_data[train_idx]
        fold_val_data = train_data[val_idx]
        
        # Train on this fold
        for epoch in range(epochs):
            epoch_loss = 0
            for i in range(len(fold_train_data)):
                sample = fold_train_data[i, 1:] / 255.0  # Normalize
                label = int(fold_train_data[i, 0])
                
                y_hat, a1, a2, z1, z2  = forwardPropgagation(sample)
                loss = backwordPropagation(y_hat, label, a1, a2, z1, z2, x=sample)
                epoch_loss += loss
            
            avg_loss = epoch_loss / len(fold_train_data)
            
            if (epoch + 1) % 2 == 0:
                print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")
        
        val_accuracy, val_loss = evaluate(fold_val_data, fold_val_data[:, 0])
        fold_accuracies.append(val_accuracy)
        fold_losses.append(val_loss)
        
        print(f"Fold {fold_num + 1} - Validation Accuracy: {val_accuracy:.4f}, Loss: {val_loss:.4f}")
        
        # Track best model
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_weights = {
                'w1': w1.copy(),
                'w2': w2.copy(),
                'w3': w3.copy(),
                'b1': b1.copy(),
                'b2': b2.copy(),
                'b3': b3.copy()
            }
            best_fold = fold_num + 1
    
    print(f"\n{'='*50}")
    print("K-Fold Cross-Validation Results")
    print(f"{'='*50}")
    print(f"Mean Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})")
    print(f"Mean Loss: {np.mean(fold_losses):.4f} (+/- {np.std(fold_losses):.4f})")
    
    for i, (acc, loss) in enumerate(zip(fold_accuracies, fold_losses)):
        print(f"Fold {i+1}: Accuracy={acc:.4f}, Loss={loss:.4f}")
    
    print(f"\n{'='*50}")
    print(f"Best Model: Fold {best_fold} with Accuracy: {best_accuracy:.4f}")
    print(f"{'='*50}")
    
    if best_weights:
        w1 = best_weights['w1']
        w2 = best_weights['w2']
        w3 = best_weights['w3']
        b1 = best_weights['b1']
        b2 = best_weights['b2']
        b3 = best_weights['b3']
        save_model(model_file_name)
        print("✓ Best model weights saved to final_model_weights.npz")
    
    return fold_accuracies, fold_losses



In [105]:
# Train the neural network using k-fold cross-validation
print("Starting K-Fold Cross-Validation Training...")
fold_accuracies, fold_losses = train_network()
print("\nTraining completed!")

Starting K-Fold Cross-Validation Training...

Fold 1/5
Epoch 2/10, Loss: 2.3664
Epoch 4/10, Loss: 2.3405
Epoch 6/10, Loss: 1.2909
Epoch 8/10, Loss: 0.4541
Epoch 10/10, Loss: 0.3224
Fold 1 - Validation Accuracy: 0.9082, Loss: 79.3774

Fold 2/5
Epoch 2/10, Loss: 2.3659
Epoch 4/10, Loss: 2.3324
Epoch 6/10, Loss: 1.2650
Epoch 8/10, Loss: 0.4421
Epoch 10/10, Loss: 0.3159
Fold 2 - Validation Accuracy: 0.9030, Loss: 79.0049

Fold 3/5
Epoch 2/10, Loss: 2.3663
Epoch 4/10, Loss: 2.3375
Epoch 6/10, Loss: 1.2833
Epoch 8/10, Loss: 0.4563
Epoch 10/10, Loss: 0.3272
Fold 3 - Validation Accuracy: 0.9033, Loss: 79.2301

Fold 4/5
Epoch 2/10, Loss: 2.3667
Epoch 4/10, Loss: 2.3420
Epoch 6/10, Loss: 1.2941
Epoch 8/10, Loss: 0.4534
Epoch 10/10, Loss: 0.3221
Fold 4 - Validation Accuracy: 0.9115, Loss: 79.6211

Fold 5/5
Epoch 2/10, Loss: 2.3670
Epoch 4/10, Loss: 2.3440
Epoch 6/10, Loss: 1.2999
Epoch 8/10, Loss: 0.4592
Epoch 10/10, Loss: 0.3317
Fold 5 - Validation Accuracy: 0.9033, Loss: 77.6444

K-Fold Cross-V

In [111]:

def predict(x: np.array):
    y_hat, _, _, _,_ = forwardPropgagation(x)
    return np.argmax(y_hat)
def test_model(test_data, test_labels=None):
    predictions = []
    correct = 0
    total = len(test_data)
    
    print(f"\nTesting on {total} samples...")
    print("="*60)
    
    for index, data in enumerate(test_data):
        pred = predict(data)
        predictions.append(pred)
        if test_labels is not None:
            actual = test_labels[index][0]
            is_correct = pred == actual
            correct += is_correct
            
            # Print every 100th sample or incorrect predictions
            if (index + 1) % 100 == 0 or not is_correct:
                status = "✓" if is_correct else "✗"
                print(f"Sample {index+1:4d}: Predicted {pred} | Actual {actual} {status}")
    print("="*60)
    
    if test_labels is not None:
        accuracy = (correct / total) * 100
        print(f"\nAccuracy: {accuracy:.2f}% ({correct}/{total} correct)")
    else:
        print(f"\nTotal predictions: {total}")
    
    return predictions


In [112]:
def save_predictions_to_csv(predictions, output_filename="predictions.csv"):
    df = pd.DataFrame({
        'ImageID': range(1, len(predictions) + 1),  # 1-indexed
        'Label': predictions
    })
    
    df.to_csv(output_filename, index=False)
    print(f"✓ Predictions saved to {output_filename}")
    print(f"  Total predictions: {len(predictions)}")
    print(f"\nFirst 5 rows:")
    print(df.head())

test_data = pd.read_csv(TEST_DATA_PATH)
test_data = np.array(test_data)
load_model(model_file_name)
prediction = test_model(test_data=test_data)
save_predictions_to_csv(predictions=prediction)


Model loaded from final_model_weights.npz

Testing on 28000 samples...

Total predictions: 28000
✓ Predictions saved to predictions.csv
  Total predictions: 28000

First 5 rows:
   ImageID  Label
0        1      2
1        2      0
2        3      9
3        4      7
4        5      3
